<a href="https://colab.research.google.com/github/LEF9701/CUDA/blob/main/vector_ad_cu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Fri May 29 05:00:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [4]:
%%writefile vector_add.cu
#include <iostream>
#include <cuda_runtime.h>

// 1. 這就是 GPU 核心函式 (CUDA Kernel)
// __global__ 關鍵字告訴編譯器：這段 code 會在 GPU (Device) 上跑，由 CPU (Host) 呼叫
__global__ void cuda_vector_add(const float* a, const float* b, float* c, int n) {
    // 透過 CUDA 的內建變數，計算當前這個 GPU 執行緒 (Thread) 負責處理哪一個陣列索引
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    // 預防執行緒數量超過陣列大小
    if (idx < n) {
        c[idx] = a[idx] + b[idx]; // 數千個執行緒同時並行做這行相加！
    }
}

int main() {
    int N = 1000000; // 一百萬筆資料
    size_t size = N * sizeof(float);

    // ======== [CPU 端配置記憶體] ========
    float *h_a = (float*)malloc(size);
    float *h_b = (float*)malloc(size);
    float *h_c = (float*)malloc(size);

    // 初始化資料
    for (int i = 0; i < N; i++) {
        h_a[i] = 1.0f;
        h_b[i] = 2.0f;
    }

    // ======== [GPU 端配置記憶體] ========
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, size);
    cudaMalloc(&d_b, size);
    cudaMalloc(&d_c, size);

    // ======== [資料：CPU 複製到 GPU] ========
    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

    // ======== [啟動 GPU 噴射加速] ========
    // 每個 Block 有 256 個執行緒，計算需要多少個 Blocks
    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

    std::cout << "正在啟動 GPU 運算... 使用 " << blocksPerGrid << " 個 Blocks，每個 Block 有 " << threadsPerBlock << " 個執行緒。" << std::endl;

    // 呼叫 CUDA 核心（<<< >>> 是 CUDA 特有的語法）
    cuda_vector_add<<<blocksPerGrid, threadsPerBlock>>>(d_a, d_b, d_c, N);

    // 等待 GPU 全部算完
    cudaDeviceSynchronize();

    // ======== [結果：GPU 複製回 CPU] ========
    cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

    // 驗證前 5 筆結果（1.0 + 2.0 應該等於 3.0）
    std::cout << "運算完成！前 5 筆結果如下：" << std::endl;
    for (int i = 0; i < 5; i++) {
        std::cout << h_a[i] << " + " << h_b[i] << " = " << h_c[i] << std::endl;
    }

    // 釋放記憶體
    free(h_a); free(h_b); free(h_c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);

    return 0;
}

Writing vector_add.cu


In [5]:
!nvcc vector_add.cu -o my_cuda_program

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [6]:
!./my_cuda_program

正在啟動 GPU 運算... 使用 3907 個 Blocks，每個 Block 有 256 個執行緒。
運算完成！前 5 筆結果如下：
1 + 2 = 3
1 + 2 = 3
1 + 2 = 3
1 + 2 = 3
1 + 2 = 3
